## VINS Dataset: Detecting UI Elements for Accessibility 
VINS dataset is an annotated dataset containing a representative collection of UI screens across two design stages: abstract wireframes and high-fidelity fully designed interfaces. All of these UIs are annotated with bounding boxes spanning different classes of UI components. We identified a total of 11 UI components with varying functionality: background images, sliding menus, pop-up windows, input fields, icons, images, texts, switches, checked views, text buttons, and page indicators. Based on our analysis and due to relatively small training instances, we combined radio buttons and checkboxes to represent the checked view class.

VINS dataset has a total of 4,800 images of UI screens of different design stages to ensure that the VINS can perform on a wider variety of design inputs. It includes the following:

- 257 images of abstract wireframes UIs

- 4,543 images of high-fidelity screens:
    - 2000 images of Android screen collected from Rico Dataset
    - 740 images of new Android screens
    - 1200 images of iphone screens
    - 603 images of UI design collected from design sharing websites


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
torchvision.disable_beta_transforms_warning()
import torchvision.transforms.v2 as transforms
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineRenderer.figure_format = 'retina'

import os
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import  utils

import zipfile
from pathlib import Path
import os
import random
import shutil

#import data_splitter

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

plt.ion()   # interactive mode

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [6]:
log_dir = 'logs'

# Load the tensorboard extension for Jupyter
%load_ext tensorboard
# Start tensorboard and tell it where to look for logs. It will auto-update every second.
%tensorboard --logdir {log_dir}

In [5]:
# Setup path to data folder
data_path = Path("/home/lurena/scratch/")
image_path = data_path / "VINS_dataset"

# If the image folder doesn't exist, download it and prepare it... 
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

    # Unzip VINS file
    with zipfile.ZipFile(data_path / "VINS_dataset.zip", "r") as zip_ref:
        print("Unzipping VINS file...") 
        zip_ref.extractall(image_path)


/home/lurena/scratch/VINS_dataset directory exists.


In [12]:
transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ConvertImageDtype(),
])

vins = torchvision.datasets.ImageFolder(image_path, transform = transform)

# Set training/test/validate sizes
train_size = 0.6
test_size = 1-(train_size+0.1) #0.1 here defines valid_size tbh
valid_size = 1-(train_size+test_size)

if train_size + test_size + valid_size > 1:
    raise ValueError("Invalid data split sizes")

train_data, test_data, valid_data = torch.utils.data.random_split(vins, [train_size, test_size, valid_size])
#print(len(vins))
#print(len(train_data), len(test_data), len(valid_data))

classes = ['bg_image', 'sld_menu','pop_up','input_fld','icon','image','txt','switch','chck_view','txt_button','page_indicator']
#radio buttons and checkboxes represented under checked view class


In [17]:
mean = []
for x, _ in vins:
    mean.append(torch.mean(x, dim=(1, 2)))
    
mean = torch.stack(mean, dim=0).mean(dim=0)
print(mean)
std = []
for x, _ in vins:
    std.append(((x - mean[:,np.newaxis,np.newaxis]) ** 2).mean(dim=(1, 2)))
std = torch.stack(std, dim=0).mean(dim=0).sqrt()
print(mean, std)

UnidentifiedImageError: cannot identify image file <_io.BufferedReader name='/home/lurena/scratch/VINS_dataset/__MACOSX/All Dataset/Android/JPEGImages/._Android_1.jpg'>

In [18]:
pwd

'/home/lurena/final_proj'